# GEMINI Pro Sense Annotation Pipeline
This notebook is a Pro-oriented copy of the Gemini pipeline. It defaults to the stable `gemini-2.5-pro` model for production use and keeps the newer `gemini-3.1-pro-preview` available as an opt-in preview model.

## Notes on model and rate limits
- Stable production default: `gemini-2.5-pro`.
- Newer Pro preview currently available: `gemini-3.1-pro-preview`.
- Paid Gemini API tiers still have rate limits; they are higher than free-tier limits, but not removed. Preview models can also have stricter limits.
- This notebook therefore uses a configurable request delay and defaults it to `0.0` seconds for paid-tier usage. If your project still hits `429` errors, increase the delay or check your active limits in AI Studio.

In [1]:
from config import GOOGLE_GENAI_API_KEY, ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR
from process_senses import process_senses_with_chain, default_build_senses_block, parse_model_output
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter
import time

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 2

# Prefer the stable Pro model for production.
# Set to True only if you explicitly want the newer preview model.
USE_PREVIEW_MODEL = False
model = "gemini-3.1-pro-preview" if USE_PREVIEW_MODEL else "gemini-2.5-pro"

# Paid tiers still have quotas; start with no fixed delay and add one only if you see 429s.
REQUEST_DELAY_SECONDS = 0.0
print(f"Using model: {model}")
print(f"Configured inter-request delay: {REQUEST_DELAY_SECONDS:.1f}s")

Using model: gemini-2.5-pro
Configured inter-request delay: 0.0s


In [2]:
# Load sense repository and sentences
from data_loader import load_sense_repo_by_round

senses_df = load_sense_repo_by_round(round_number=ROUND)

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

# Human-readable chunk selection
chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

# Choose which chunk to process here (0-based)
chunk_idx = 0
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

# Testing flag: when True, take only a small slice; when False, keep full chunk
# By default, use chunk boundaries for filenames
begin, end = chunk_begin, chunk_end

test = True
if test:
    # Use a predictable small slice for quick dry runs
    tb, te = 0, 20
    sentences = sentences[tb:te]
    # Human-facing indices in filenames (1-based start, inclusive end)
    begin, end = tb + 1, te

Available chunks (index, begin, end, file): [(0, 1, 500, 'sr-elexis-WSD_0001_0500.tsv'), (1, 501, 1000, 'sr-elexis-WSD_0501_1000.tsv'), (2, 1001, 1500, 'sr-elexis-WSD_1001_1500.tsv'), (3, 1501, 2000, 'sr-elexis-WSD_1501_2000.tsv'), (4, 2001, 2024, 'sr-elexis-WSD_2001_2024.tsv')]
Using chunk #0: sr-elexis-WSD_0001_0500.tsv -> (1, 500)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate

# Compose Gemini Pro LLM chain
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  \"sense_id\": \"<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>\",
  \"explanation\": \"<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>\"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW_SENSE'; nikada ne smete izmišljati druge ID-ove.
"""

user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

llm = ChatGoogleGenerativeAI(
    temperature=0,
    google_api_key=GOOGLE_GENAI_API_KEY,
    model=model,
)
# Set model origin for traceability
ORIGIN_LLM = f"GeminiPro_{model}"

parser = StrOutputParser()
chain = prompt | llm | parser

In [ ]:
# Annotate senses using the shared utility
start_time = time.time()
sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM,
    build_senses_block=default_build_senses_block,
    parse_json_response_clean=parse_model_output,
    time_delay=REQUEST_DELAY_SECONDS
)
end_time = time.time()
print(f"Processed sentences {begin} to {end} in {end_time - start_time:.2f} seconds.")

In [ ]:
# Save outputs (add ROUND_SUFFIX for versioning)
ROUND_SUFFIX = "_test"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

In [ ]:
# Write Inception-compatible output (for annotation import)
incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

## Log processing time and completion

In [ ]:
with open("gemini_pro.log", "a", encoding="utf-8") as f:
    f.write(f"Processed sentences {begin} to {end}\n")
    f.write(f"Model: {model}\n")
    f.write(f"Delay: {REQUEST_DELAY_SECONDS:.1f}s\n")
    f.write(f"Gemini Pro run took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")